# 04 - Knowledge Base
Create an OpenSearch Serverless collection, Bedrock Knowledge Base with S3 data source, and test RAG retrieval.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]
EMBEDDING_MODEL = config["EMBEDDING_MODEL_ID"]
EMBEDDING_DIM = config["EMBEDDING_DIMENSIONS"]

tracker = ResourceTracker()
s3 = boto3.client("s3", region_name=REGION)
oss = boto3.client("opensearchserverless", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)
iam = boto3.client("iam")

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Create S3 Bucket for Documents

In [ ]:
BUCKET_NAME = f"{PREFIX}-documents-{ACCOUNT_ID}-{REGION}"

try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"\u2713 Bucket already exists: {BUCKET_NAME}")
except:
    print(f"\u231b Creating S3 bucket: {BUCKET_NAME}...")
    create_kwargs = {"Bucket": BUCKET_NAME}
    if REGION != "us-east-1":
        create_kwargs["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**create_kwargs)
    tracker.add_s3_bucket(BUCKET_NAME)
    print(f"\u2713 Bucket created: {BUCKET_NAME}")

## Step 3: Upload Sample Documents

In [ ]:
sample_docs = [
    {
        "key": "docs/kubernetes-migration.md",
        "content": """# Kubernetes Migration Playbook\n\nThis guide covers migrating monolithic applications to Kubernetes.\n\n## Prerequisites\n- Container runtime (Docker/containerd)\n- kubectl configured\n- Cluster with 3+ nodes\n\n## Steps\n1. Containerize the application with multi-stage Dockerfiles\n2. Define resource requests and limits\n3. Create Deployment and Service manifests\n4. Configure Horizontal Pod Autoscaler\n5. Set up Ingress with TLS termination\n\n## Best Practices\n- Use namespaces for isolation\n- Implement readiness and liveness probes\n- Store secrets in AWS Secrets Manager\n- Use Karpenter for node autoscaling"""
    },
    {
        "key": "docs/ml-pipeline-architecture.md",
        "content": """# ML Pipeline Architecture\n\nEnd-to-end machine learning pipeline using SageMaker and Step Functions.\n\n## Components\n- Data ingestion: S3 + Glue ETL\n- Feature store: SageMaker Feature Store\n- Training: SageMaker Training Jobs\n- Evaluation: Custom metrics + SageMaker Experiments\n- Deployment: Multi-model endpoints\n\n## Model Registry\nAll models versioned in SageMaker Model Registry with approval workflows.\n\n## Monitoring\n- Model drift detection with SageMaker Model Monitor\n- A/B testing via traffic splitting\n- CloudWatch custom metrics for business KPIs"""
    },
    {
        "key": "docs/zero-trust-auth.md",
        "content": """# Zero Trust Authentication Design\n\nImplementing zero trust security for microservices.\n\n## Principles\n- Never trust, always verify\n- Least privilege access\n- Assume breach\n\n## Implementation\n1. mTLS between all services (Istio service mesh)\n2. JWT validation at every service boundary\n3. Short-lived tokens (15 min expiry)\n4. Device trust scoring\n5. Continuous authorization checks\n\n## Token Architecture\n- Identity Provider: AWS Cognito + OIDC federation\n- Access tokens: RS256 signed JWTs\n- Refresh tokens: stored in DynamoDB with rotation\n- Service tokens: AWS IAM roles for service accounts"""
    },
]

for doc in sample_docs:
    s3.put_object(Bucket=BUCKET_NAME, Key=doc["key"], Body=doc["content"].encode())
    print(f"  \u2713 Uploaded: {doc['key']}")

print(f"\u2713 {len(sample_docs)} documents uploaded to S3")

## Step 4: Create OpenSearch Serverless Encryption Policy

In [ ]:
COLLECTION_NAME = f"{PREFIX}-kb-collection"

# Encryption policy
enc_policy_name = f"{PREFIX}-kb-enc-policy"
try:
    oss.get_security_policy(name=enc_policy_name, type="encryption")
    print(f"\u2713 Encryption policy exists: {enc_policy_name}")
except oss.exceptions.ResourceNotFoundException:
    enc_policy = json.dumps({
        "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
        "AWSOwnedKey": True
    })
    oss.create_security_policy(
        name=enc_policy_name,
        type="encryption",
        policy=enc_policy
    )
    tracker.add_opensearch_policy(enc_policy_name, "encryption")
    print(f"\u2713 Encryption policy created: {enc_policy_name}")

## Step 5: Create OpenSearch Serverless Network Policy

In [ ]:
net_policy_name = f"{PREFIX}-kb-net-policy"
try:
    oss.get_security_policy(name=net_policy_name, type="network")
    print(f"\u2713 Network policy exists: {net_policy_name}")
except oss.exceptions.ResourceNotFoundException:
    net_policy = json.dumps([{
        "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
                  {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}],
        "AllowFromPublic": True
    }])
    oss.create_security_policy(
        name=net_policy_name,
        type="network",
        policy=net_policy
    )
    tracker.add_opensearch_policy(net_policy_name, "network")
    print(f"\u2713 Network policy created: {net_policy_name}")

## Step 6: Create OpenSearch Serverless Collection

In [ ]:
# Check if collection exists
collections = oss.list_collections(
    collectionFilters={"name": COLLECTION_NAME}
)["collectionSummaries"]

if collections:
    collection_id = collections[0]["id"]
    collection_arn = collections[0]["arn"]
    print(f"\u2713 Collection already exists: {collection_id}")
else:
    print(f"\u231b Creating collection: {COLLECTION_NAME}...")
    response = oss.create_collection(
        name=COLLECTION_NAME,
        type="VECTORSEARCH",
        description=f"OrgIQ knowledge base vector store"
    )
    collection_id = response["createCollectionDetail"]["id"]
    collection_arn = response["createCollectionDetail"]["arn"]
    tracker.add_opensearch_collection(collection_arn, COLLECTION_NAME)
    print(f"\u2713 Collection creation initiated: {collection_id}")

## Step 7: Wait for Collection to Become Active

In [ ]:
print("\u231b Waiting for collection to become ACTIVE...")
while True:
    coll_detail = oss.batch_get_collection(ids=[collection_id])
    status = coll_detail["collectionDetails"][0]["status"]
    if status == "ACTIVE":
        collection_endpoint = coll_detail["collectionDetails"][0]["collectionEndpoint"]
        print(f"\u2713 Collection is ACTIVE")
        print(f"  Endpoint: {collection_endpoint}")
        break
    elif status == "FAILED":
        raise RuntimeError("Collection creation failed")
    else:
        print(f"  Status: {status}, waiting 30s...")
        time.sleep(30)

## Step 8: Create Data Access Policy

In [ ]:
access_policy_name = f"{PREFIX}-kb-access-policy"

# Get current caller identity for access
sts = boto3.client("sts")
caller_arn = sts.get_caller_identity()["Arn"]

# KB role ARN (will be created next)
kb_role_name = f"{PREFIX}-kb-role"
kb_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{kb_role_name}"

access_policy = json.dumps([{
    "Rules": [
        {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"],
         "Permission": ["aoss:CreateCollectionItems", "aoss:DeleteCollectionItems",
                        "aoss:UpdateCollectionItems", "aoss:DescribeCollectionItems"]},
        {"ResourceType": "index", "Resource": [f"index/{COLLECTION_NAME}/*"],
         "Permission": ["aoss:CreateIndex", "aoss:DeleteIndex", "aoss:UpdateIndex",
                        "aoss:DescribeIndex", "aoss:ReadDocument", "aoss:WriteDocument"]}
    ],
    "Principal": [caller_arn, kb_role_arn]
}])

try:
    oss.get_access_policy(name=access_policy_name, type="data")
    oss.update_access_policy(name=access_policy_name, type="data", policy=access_policy, policyVersion="2")
    print(f"\u2713 Access policy updated: {access_policy_name}")
except oss.exceptions.ResourceNotFoundException:
    oss.create_access_policy(name=access_policy_name, type="data", policy=access_policy)
    tracker.add_opensearch_policy(access_policy_name, "access")
    print(f"\u2713 Access policy created: {access_policy_name}")

## Step 9: Create IAM Role for Knowledge Base

In [ ]:
trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {
            "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
            "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/*"}
        }
    }]
})

try:
    iam.get_role(RoleName=kb_role_name)
    print(f"\u2713 IAM role already exists: {kb_role_name}")
except iam.exceptions.NoSuchEntityException:
    iam.create_role(
        RoleName=kb_role_name,
        AssumeRolePolicyDocument=trust_policy,
        Description="Role for OrgIQ Bedrock Knowledge Base"
    )
    
    # Attach permissions
    kb_policy = json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["bedrock:InvokeModel"],
             "Resource": f"arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}"},
            {"Effect": "Allow", "Action": ["s3:GetObject", "s3:ListBucket"],
             "Resource": [f"arn:aws:s3:::{BUCKET_NAME}", f"arn:aws:s3:::{BUCKET_NAME}/*"]},
            {"Effect": "Allow", "Action": ["aoss:APIAccessAll"],
             "Resource": collection_arn}
        ]
    })
    iam.put_role_policy(RoleName=kb_role_name, PolicyName=f"{PREFIX}-kb-policy", PolicyDocument=kb_policy)
    tracker.add_iam_role(kb_role_arn, kb_role_name)
    print(f"\u2713 IAM role created: {kb_role_name}")
    print("\u231b Waiting 10s for IAM propagation...")
    time.sleep(10)

## Step 10: Create Bedrock Knowledge Base

In [ ]:
KB_NAME = f"{PREFIX}-knowledge-base"

# Check if KB already exists
existing_kbs = bedrock_agent.list_knowledge_bases()
existing_kb = next(
    (kb for kb in existing_kbs["knowledgeBaseSummaries"] if kb["name"] == KB_NAME),
    None
)

if existing_kb:
    kb_id = existing_kb["knowledgeBaseId"]
    print(f"\u2713 Knowledge Base already exists: {kb_id}")
else:
    print(f"\u231b Creating Knowledge Base: {KB_NAME}...")
    response = bedrock_agent.create_knowledge_base(
        name=KB_NAME,
        description="OrgIQ organizational knowledge base for RAG retrieval",
        roleArn=kb_role_arn,
        knowledgeBaseConfiguration={
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {
                "embeddingModelArn": f"arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}",
                "embeddingModelConfiguration": {
                    "bedrockEmbeddingModelConfiguration": {
                        "dimensions": EMBEDDING_DIM
                    }
                }
            }
        },
        storageConfiguration={
            "type": "OPENSEARCH_SERVERLESS",
            "opensearchServerlessConfiguration": {
                "collectionArn": collection_arn,
                "vectorIndexName": f"{PREFIX}-kb-index",
                "fieldMapping": {
                    "vectorField": "embedding",
                    "textField": "text",
                    "metadataField": "metadata"
                }
            }
        }
    )
    kb_id = response["knowledgeBase"]["knowledgeBaseId"]
    tracker.add_knowledge_base(kb_id, KB_NAME)
    print(f"\u2713 Knowledge Base created: {kb_id}")

## Step 11: Create S3 Data Source

In [ ]:
DS_NAME = f"{PREFIX}-docs-source"

# Check if data source already exists
existing_ds = bedrock_agent.list_data_sources(knowledgeBaseId=kb_id)
existing_source = next(
    (ds for ds in existing_ds["dataSourceSummaries"] if ds["name"] == DS_NAME),
    None
)

if existing_source:
    ds_id = existing_source["dataSourceId"]
    print(f"\u2713 Data source already exists: {ds_id}")
else:
    print(f"\u231b Creating data source: {DS_NAME}...")
    response = bedrock_agent.create_data_source(
        knowledgeBaseId=kb_id,
        name=DS_NAME,
        description="S3 bucket with organizational documents",
        dataSourceConfiguration={
            "type": "S3",
            "s3Configuration": {
                "bucketArn": f"arn:aws:s3:::{BUCKET_NAME}",
                "inclusionPrefixes": ["docs/"]
            }
        },
        vectorIngestionConfiguration={
            "chunkingConfiguration": {
                "chunkingStrategy": "FIXED_SIZE",
                "fixedSizeChunkingConfiguration": {
                    "maxTokens": 512,
                    "overlapPercentage": 20
                }
            }
        }
    )
    ds_id = response["dataSource"]["dataSourceId"]
    tracker.add_data_source(kb_id, ds_id)
    print(f"\u2713 Data source created: {ds_id}")

## Step 12: Start Ingestion Job

In [ ]:
print("\u231b Starting ingestion job...")
ingestion = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=ds_id
)
job_id = ingestion["ingestionJob"]["ingestionJobId"]
print(f"  Job ID: {job_id}")

# Wait for ingestion to complete
while True:
    job_status = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=job_id
    )
    status = job_status["ingestionJob"]["status"]
    if status == "COMPLETE":
        stats = job_status["ingestionJob"].get("statistics", {})
        print(f"\u2713 Ingestion complete")
        print(f"  Documents scanned: {stats.get('numberOfDocumentsScanned', 'N/A')}")
        print(f"  Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 'N/A')}")
        break
    elif status == "FAILED":
        reason = job_status["ingestionJob"].get("failureReasons", ["Unknown"])
        raise RuntimeError(f"Ingestion failed: {reason}")
    else:
        print(f"  Status: {status}, waiting 15s...")
        time.sleep(15)

## Step 13: Test RAG Retrieval

In [ ]:
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

# Test retrieval
query = "How do I migrate applications to Kubernetes?"
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": query},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 3
        }
    }
)

print(f"Query: '{query}'")
print(f"\nRetrieved {len(response['retrievalResults'])} results:")
for i, result in enumerate(response["retrievalResults"], 1):
    score = result.get("score", 0)
    text_preview = result["content"]["text"][:150]
    print(f"\n  Result {i} (score: {score:.4f}):")
    print(f"    {text_preview}...")

print("\n\u2713 RAG retrieval working")

## Step 14: Test Retrieve and Generate (RAG)

In [ ]:
# Full RAG: retrieve + generate answer
rag_query = "What are the best practices for zero trust authentication?"
response = bedrock_agent_runtime.retrieve_and_generate(
    input={"text": rag_query},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": kb_id,
            "modelArn": f"arn:aws:bedrock:{REGION}::foundation-model/{config['MODEL_ID']}"
        }
    }
)

answer = response["output"]["text"]
citations = response.get("citations", [])

print(f"Query: '{rag_query}'")
print(f"\nAnswer:\n{answer}")
print(f"\nCitations: {len(citations)} sources referenced")
print("\n\u2713 Retrieve-and-generate working")

## Step 15: Save Configuration

In [ ]:
config["KB_ID"] = kb_id
config["KB_DATA_SOURCE_ID"] = ds_id
config["COLLECTION_ID"] = collection_id
config["COLLECTION_ARN"] = collection_arn
config["S3_BUCKET"] = BUCKET_NAME

Path("config.json").write_text(json.dumps(config, indent=2))
print(f"\u2713 Config saved")
print(f"  KB_ID: {kb_id}")
print(f"  KB_DATA_SOURCE_ID: {ds_id}")
print(f"  S3_BUCKET: {BUCKET_NAME}")

## Summary

- [x] S3 bucket created with sample documents
- [x] OpenSearch Serverless collection (VECTORSEARCH type)
- [x] Security policies (encryption, network, access)
- [x] IAM role for Bedrock Knowledge Base
- [x] Bedrock Knowledge Base with Titan V2 embeddings (1024d)
- [x] S3 data source with chunking configuration
- [x] Ingestion completed successfully
- [x] RAG retrieval and retrieve-and-generate validated

**Next:** [05-guardrails.ipynb](05-guardrails.ipynb) — Create Bedrock Guardrails for content safety